# Milestone 3 Implementation: Advanced Feature Engineering & XGBoost

**Author:** Saksham Kapoor  
**Objective:** Implement 5 new feature categories, upgrade to XGBoost, and enhance anomaly explanations

## Overview

This notebook implements:
1. **Rolling Statistics** - Temporal memory for trend detection
2. **Business Efficiency Ratios** - Interpretable KPIs
3. **Lag Features** - Daily/weekly seasonality
4. **Network Context** - Blast radius from transaction graph
5. **Multi-Metric Signals** - Combined anomaly detection
6. **XGBoost Model** - State-of-the-art tabular ML
7. **Enhanced Explanations** - Severity levels and recommendations

## Baseline Comparison

**Milestone 2 Best Model (KNN):**
- PR-AUC: 0.9925
- F1-Score: 0.9943
- Precision: 1.0000
- Recall: 0.9887
- Accuracy: 0.9990


In [1]:
# Imports
import json
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
from datetime import datetime
from itertools import islice
import warnings
warnings.filterwarnings('ignore')

# ML imports
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully")
print(f"XGBoost version: {xgb.__version__}")
print(f"Polars version: {pl.__version__}")


Libraries imported successfully
XGBoost version: 3.1.1
Polars version: 1.35.2


## Step 1: Data Loading and Initial Processing


In [2]:
# Load daily metrics using Polars (much faster!)
daily_path = Path("../daily_metrics.jsonl")
print(f"Loading daily_metrics.jsonl from {daily_path}...")

# Load JSONL directly with Polars
df_daily = pl.read_ndjson(str(daily_path))

# Rename columns to match expected format
df_daily = df_daily.rename({
    'tenant/id': 'tenant_id',
    'app/id': 'app_id',
    'app/name': 'app_name',
    'daily/time': 'date',
    'daily/metric': 'metric',
    'daily/value': 'value'
})

# Debug: Check date format
print(f"\nSample date values (first 5):")
print(df_daily.select('date').head(5))
print(f"\nDate column type: {df_daily['date'].dtype}")

# Convert date to datetime - handle both formats: 'YYYY-MM-DDTHH:MM:SS' or 'YYYY-MM-DD'
# Use conditional parsing: if contains 'T', parse as datetime; otherwise as date
df_daily = df_daily.with_columns([
    pl.when(pl.col('date').str.contains('T'))
    .then(
        # Parse as datetime format
        pl.col('date').str.strptime(pl.Datetime, format='%Y-%m-%dT%H:%M:%S', strict=False)
    )
    .otherwise(
        # Parse as date format and cast to datetime
        pl.col('date').str.strptime(pl.Date, format='%Y-%m-%d', strict=False).cast(pl.Datetime)
    )
    .alias('date')
])

# Convert value to float
df_daily = df_daily.with_columns([
    pl.col('value').cast(pl.Float64, strict=False).alias('value')
])

print(f"\nLoaded {len(df_daily):,} daily metric rows")
print(f"Date range: {df_daily['date'].min()} to {df_daily['date'].max()}")
print(f"Unique apps: {df_daily['app_id'].n_unique()}")
print(f"Unique metrics: {df_daily['metric'].unique().to_list()}")


Loading daily_metrics.jsonl from ..\daily_metrics.jsonl...

Sample date values (first 5):
shape: (5, 1)
┌─────────────────────┐
│ date                │
│ ---                 │
│ str                 │
╞═════════════════════╡
│ 2024-06-01T00:00:00 │
│ 2024-06-01T00:00:00 │
│ 2024-06-01T00:00:00 │
│ 2024-06-01T00:00:00 │
│ 2024-06-01T00:00:00 │
└─────────────────────┘

Date column type: String

Loaded 666,855 daily metric rows
Date range: 2024-06-01 00:00:00 to 2025-05-31 00:00:00
Unique apps: 87
Unique metrics: ['requests_per_hour', 'outage_duration', 'data_sent', 'outage_cost', 'cost', 'data_used_per_received', 'requests_made', 'cost_per_request_made', 'value', 'data_sent_per_received', 'outage_count', 'cost_per_request_received', 'requests_received', 'value_per_cost', 'outage_efficiency', 'outage_frequency', 'data_per_request', 'outage_severity', 'requests_per_business_hour', 'outage_impact', 'data_used']


In [3]:
# Pivot daily metrics to wide format (one row per app per day) using Polars
df_daily_wide = df_daily.pivot(
    index=['tenant_id', 'app_id', 'app_name', 'date'],
    columns='metric',
    values='value',
    aggregate_function='first'
)

# Fill missing values with 0
df_daily_wide = df_daily_wide.fill_null(0)

print(f"Pivoted shape: {df_daily_wide.shape}")
print(f"Columns: {df_daily_wide.columns}")
df_daily_wide.head()


Pivoted shape: (31755, 25)
Columns: ['tenant_id', 'app_id', 'app_name', 'date', 'cost', 'value', 'data_used', 'data_sent', 'requests_made', 'requests_received', 'cost_per_request_made', 'cost_per_request_received', 'data_per_request', 'value_per_cost', 'requests_per_hour', 'requests_per_business_hour', 'data_sent_per_received', 'data_used_per_received', 'outage_count', 'outage_duration', 'outage_impact', 'outage_cost', 'outage_frequency', 'outage_severity', 'outage_efficiency']


tenant_id,app_id,app_name,date,cost,value,data_used,data_sent,requests_made,requests_received,cost_per_request_made,cost_per_request_received,data_per_request,value_per_cost,requests_per_hour,requests_per_business_hour,data_sent_per_received,data_used_per_received,outage_count,outage_duration,outage_impact,outage_cost,outage_frequency,outage_severity,outage_efficiency
str,str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""DEMO""","""SELENE""","""Selene Customer Warehouse""",2024-06-01 00:00:00,158758.25,4646.1,4646.1,4646.1,5490.0,1830.0,28.917714,86.753142,0.846284,0.029265,228.75,296.8,2.538852,2.538852,1.0,135.0,0.0,0.0,1.0,135.0,0.0
"""DEMO""","""AWS""","""Amazon Web Services""",2024-06-01 00:00:00,0.0,1342.65,0.0,1342.65,0.0,4599.0,0.0,0.0,0.0,0.0,0.0,185.5,0.291944,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""DEMO""","""DBRCKS""","""Databricks""",2024-06-01 00:00:00,0.0,3673.2,0.0,3673.2,0.0,3424.0,0.0,0.0,0.0,0.0,0.0,138.5,1.07278,0.0,0.0,0.0,95.0,3938.35,0.0,0.0,3938.35
"""DEMO""","""SNWFLK""","""Snowflake""",2024-06-01 00:00:00,0.0,2185.65,0.0,2185.65,0.0,2259.0,0.0,0.0,0.0,0.0,0.0,90.6,0.96753,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""DEMO""","""FRDETCT""","""Fraud Detection System""",2024-06-01 00:00:00,11736.2,176.25,268.05,176.25,192.0,41.0,61.126042,286.24878,0.917969,0.015018,8.0,11.6,4.29878,6.537805,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# Load and aggregate transactions into hourly windows using Polars (much faster!)
transactions_path = Path("../transactions.jsonl")
print(f"Loading transactions.jsonl from {transactions_path}...")

# Load transactions with Polars - this is MUCH faster than line-by-line parsing
df_txn_raw = pl.read_ndjson(str(transactions_path))

print(f"Loaded {len(df_txn_raw):,} transactions")

# Process consumer perspective aggregations
df_txn_consumer = (
    df_txn_raw
    .with_columns([
        pl.col('transaction/time').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S').dt.truncate('1h').alias('hour'),
        (pl.col('transaction/response') != 'success').cast(pl.Int64).alias('is_failure'),
        pl.col('transaction/cost').cast(pl.Float64),
        pl.col('transaction/data').cast(pl.Float64)
    ])
    .group_by(['transaction/consumer/id', 'hour'])
    .agg([
        pl.len().alias('req_count'),
        pl.sum('transaction/cost').alias('cost_sum'),
        pl.mean('transaction/cost').alias('cost_mean'),
        pl.sum('transaction/data').alias('data_sum'),
        pl.mean('transaction/data').alias('data_mean'),
        pl.mean('is_failure').alias('error_rate')
    ])
    .rename({'transaction/consumer/id': 'app_id'})
)

print(f"Consumer aggregations: {len(df_txn_consumer):,} unique app-hour pairs")
print(f"Sample of consumer aggregations:")
print(df_txn_consumer.head())


Loading transactions.jsonl from ..\transactions.jsonl...
Loaded 7,254,656 transactions
Consumer aggregations: 531,075 unique app-hour pairs
Sample of consumer aggregations:
shape: (5, 8)
┌─────────┬────────────────┬───────────┬──────────┬────────────┬──────────┬───────────┬────────────┐
│ app_id  ┆ hour           ┆ req_count ┆ cost_sum ┆ cost_mean  ┆ data_sum ┆ data_mean ┆ error_rate │
│ ---     ┆ ---            ┆ ---       ┆ ---      ┆ ---        ┆ ---      ┆ ---       ┆ ---        │
│ str     ┆ datetime[μs]   ┆ u32       ┆ f64      ┆ f64        ┆ f64      ┆ f64       ┆ f64        │
╞═════════╪════════════════╪═══════════╪══════════╪════════════╪══════════╪═══════════╪════════════╡
│ HMFRP   ┆ 2024-09-11     ┆ 69        ┆ 1336.3   ┆ 19.366667  ┆ 9.9      ┆ 0.143478  ┆ 0.0        │
│         ┆ 06:00:00       ┆           ┆          ┆            ┆          ┆           ┆            │
│ FINPLAN ┆ 2024-07-02     ┆ 3         ┆ 170.15   ┆ 56.716667  ┆ 3.45     ┆ 1.15      ┆ 0.0        │
│    

In [5]:
# Use consumer aggregations (already computed with Polars)
df_txn = df_txn_consumer

print(f"Transaction aggregates shape: {df_txn.shape}")
print(f"Columns: {df_txn.columns}")
df_txn.head()


Transaction aggregates shape: (531075, 8)
Columns: ['app_id', 'hour', 'req_count', 'cost_sum', 'cost_mean', 'data_sum', 'data_mean', 'error_rate']


app_id,hour,req_count,cost_sum,cost_mean,data_sum,data_mean,error_rate
str,datetime[μs],u32,f64,f64,f64,f64,f64
"""HMFRP""",2024-09-11 06:00:00,69,1336.3,19.366667,9.9,0.143478,0.0
"""FINPLAN""",2024-07-02 22:00:00,3,170.15,56.716667,3.45,1.15,0.0
"""FINRPT""",2025-03-07 16:00:00,3,307.1,102.366667,4.65,1.55,0.0
"""EXCS""",2024-09-10 10:00:00,10,327.85,32.785,9.45,0.945,0.0
"""YGSCPA""",2024-10-01 19:00:00,2,37.35,18.675,0.3,0.15,0.0


In [6]:
# Merge daily metrics with hourly transaction aggregates using Polars
# Convert daily date to hour (use midnight hour for daily aggregation)
df_daily_wide = df_daily_wide.with_columns([
    pl.col('date').dt.truncate('1h').alias('hour')
])

# Merge on app_id and hour
df = df_daily_wide.join(
    df_txn,
    on=['app_id', 'hour'],
    how='left',
    suffix='_hourly'
)

# Fill missing transaction metrics with 0
txn_cols = ['req_count', 'cost_sum', 'cost_mean', 'data_sum', 'data_mean', 'error_rate']
for col in txn_cols:
    if col in df.columns:
        df = df.with_columns(pl.col(col).fill_null(0))

print(f"Merged dataframe shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()


Merged dataframe shape: (31755, 32)
Date range: 2024-06-01 00:00:00 to 2025-05-31 00:00:00


tenant_id,app_id,app_name,date,cost,value,data_used,data_sent,requests_made,requests_received,cost_per_request_made,cost_per_request_received,data_per_request,value_per_cost,requests_per_hour,requests_per_business_hour,data_sent_per_received,data_used_per_received,outage_count,outage_duration,outage_impact,outage_cost,outage_frequency,outage_severity,outage_efficiency,hour,req_count,cost_sum,cost_mean,data_sum,data_mean,error_rate
str,str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,datetime[μs],u32,f64,f64,f64,f64,f64
"""DEMO""","""SELENE""","""Selene Customer Warehouse""",2024-06-01 00:00:00,158758.25,4646.1,4646.1,4646.1,5490.0,1830.0,28.917714,86.753142,0.846284,0.029265,228.75,296.8,2.538852,2.538852,1.0,135.0,0.0,0.0,1.0,135.0,0.0,2024-06-01 00:00:00,213,6332.9,29.731925,176.55,0.828873,0.0
"""DEMO""","""AWS""","""Amazon Web Services""",2024-06-01 00:00:00,0.0,1342.65,0.0,1342.65,0.0,4599.0,0.0,0.0,0.0,0.0,0.0,185.5,0.291944,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-01 00:00:00,0,0.0,0.0,0.0,0.0,0.0
"""DEMO""","""DBRCKS""","""Databricks""",2024-06-01 00:00:00,0.0,3673.2,0.0,3673.2,0.0,3424.0,0.0,0.0,0.0,0.0,0.0,138.5,1.07278,0.0,0.0,0.0,95.0,3938.35,0.0,0.0,3938.35,2024-06-01 00:00:00,0,0.0,0.0,0.0,0.0,0.0
"""DEMO""","""SNWFLK""","""Snowflake""",2024-06-01 00:00:00,0.0,2185.65,0.0,2185.65,0.0,2259.0,0.0,0.0,0.0,0.0,0.0,90.6,0.96753,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-01 00:00:00,0,0.0,0.0,0.0,0.0,0.0
"""DEMO""","""FRDETCT""","""Fraud Detection System""",2024-06-01 00:00:00,11736.2,176.25,268.05,176.25,192.0,41.0,61.126042,286.24878,0.917969,0.015018,8.0,11.6,4.29878,6.537805,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-01 00:00:00,3,224.1,74.7,3.75,1.25,0.0


## Step 2: Feature Engineering

### Category 1: Rolling Statistics (Temporal Memory)


In [7]:
# Sort by app and date for rolling calculations
df = df.sort(['app_id', 'date'])

# Rolling 7-day average (using daily data, so 7 days = 7 rows)
# In Polars, rolling_mean() doesn't support min_periods, so we'll use window_size=7
# For partial windows at the start, Polars will compute mean with available values
df = df.with_columns([
    pl.col('cost').rolling_mean(window_size=7).over('app_id').alias('cost_7day_avg')
])

df = df.with_columns([
    ((pl.col('cost') - pl.col('cost_7day_avg')) / (pl.col('cost_7day_avg') + 1)).alias('cost_deviation_from_week'),
    (pl.col('date').dt.weekday() >= 6).alias('is_weekend'),  # 6=Saturday, 7=Sunday
    pl.col('hour').dt.hour().alias('hour_of_day'),
    pl.col('date').dt.weekday().alias('day_of_week')
])

print("Rolling statistics features created")
print(f"Features added: cost_7day_avg, cost_deviation_from_week, is_weekend, hour_of_day, day_of_week")


Rolling statistics features created
Features added: cost_7day_avg, cost_deviation_from_week, is_weekend, hour_of_day, day_of_week


### Category 2: Business Efficiency Ratios


In [8]:
# Business efficiency ratios
df = df.with_columns([
    (pl.col('cost_sum') / (pl.col('req_count') + 1)).alias('cost_per_request'),
    (pl.col('data_sum') / (pl.col('req_count') + 1)).alias('data_per_request'),
    (pl.col('value') / (pl.col('cost') + 1)).alias('value_to_cost_ratio'),
    (pl.col('data_used') / (pl.col('data_sent') + 0.001)).alias('data_utilization')
])

print("Business efficiency ratios created")
print(f"Features added: cost_per_request, data_per_request, value_to_cost_ratio, data_utilization")


Business efficiency ratios created
Features added: cost_per_request, data_per_request, value_to_cost_ratio, data_utilization


### Category 3: Lag Features (Daily/Weekly Seasonality)


In [9]:
# 24 and 48-hour comparisons (using daily data, so shift by 1 and 2 days)
df = df.with_columns([
    pl.col('cost').shift(1).over('app_id').alias('cost_24h_ago'),
    pl.col('cost').shift(2).over('app_id').alias('cost_48h_ago'),
    pl.col('error_rate').shift(1).over('app_id').alias('error_rate_24h_ago')
])

df = df.with_columns([
    ((pl.col('cost') - pl.col('cost_24h_ago')) / (pl.col('cost_24h_ago') + 1)).alias('cost_pct_change_vs_24h_ago'),
    ((pl.col('cost') - pl.col('cost_48h_ago')) / (pl.col('cost_48h_ago') + 1)).alias('cost_pct_change_vs_48h_ago'),
    ((pl.col('error_rate') - pl.col('error_rate_24h_ago')) / (pl.col('error_rate_24h_ago') + 0.001)).alias('error_rate_pct_change')
])

print("Lag features created")
print(f"Features added: cost_24h_ago, cost_pct_change_vs_24h_ago, cost_48h_ago, cost_pct_change_vs_48h_ago")


Lag features created
Features added: cost_24h_ago, cost_pct_change_vs_24h_ago, cost_48h_ago, cost_pct_change_vs_48h_ago


### Category 4: Network Context (Blast Radius)

Building network features from transaction graph data.


In [10]:
# Build network context from transactions using Polars (much faster!)
# Sample first 1M transactions for speed, or use all if smaller
df_txn_sample = df_txn_raw.head(1_000_000) if len(df_txn_raw) > 1_000_000 else df_txn_raw

# Process consumer perspective (unique suppliers per consumer-hour)
df_network_consumer = (
    df_txn_sample
    .with_columns([
        pl.col('transaction/time').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S').dt.truncate('1h').alias('hour'),
        (pl.col('transaction/response') != 'success').cast(pl.Int64).alias('is_failure')
    ])
    .group_by(['transaction/consumer/id', 'hour'])
    .agg([
        pl.col('transaction/supplier/id').n_unique().alias('unique_suppliers_count'),
        pl.mean('is_failure').alias('supplier_failure_exposure')
    ])
    .rename({'transaction/consumer/id': 'app_id'})
)

# Process supplier perspective (unique consumers per supplier-hour)
df_network_supplier = (
    df_txn_sample
    .with_columns([
        pl.col('transaction/time').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S').dt.truncate('1h').alias('hour')
    ])
    .group_by(['transaction/supplier/id', 'hour'])
    .agg([
        pl.col('transaction/consumer/id').n_unique().alias('unique_consumers_count')
    ])
    .rename({'transaction/supplier/id': 'app_id'})
)

# Combine network features
df_network = (
    df_network_consumer
    .join(df_network_supplier, on=['app_id', 'hour'], how='outer', coalesce=True)
    .fill_null(0)
)

# Merge with main dataframe
df = df.join(
    df_network,
    on=['app_id', 'hour'],
    how='left'
)

# Fill missing network features with 0
df = df.with_columns([
    pl.col('unique_suppliers_count').fill_null(0),
    pl.col('unique_consumers_count').fill_null(0),
    pl.col('supplier_failure_exposure').fill_null(0)
])

# Downstream impact score
df = df.with_columns([
    (pl.col('unique_consumers_count') * pl.col('error_rate')).alias('downstream_impact_score')
])

print("Network context features created")
print(f"Features added: unique_suppliers_count, unique_consumers_count, supplier_failure_exposure, downstream_impact_score")


Network context features created
Features added: unique_suppliers_count, unique_consumers_count, supplier_failure_exposure, downstream_impact_score


### Category 5: App-Normalized Z-Scores and Multi-Metric Signals


In [11]:
# Compute app-normalized z-scores for key metrics
numeric_cols = ['cost', 'cost_mean', 'error_rate', 'value', 'value_to_cost_ratio', 'cost_per_request']

for col in numeric_cols:
    if col in df.columns:
        # App-normalized z-score using Polars
        mean_col = pl.col(col).mean().over('app_id')
        std_col = pl.col(col).std().over('app_id')
        df = df.with_columns([
            ((pl.col(col) - mean_col) / (std_col + 1e-8)).alias(f'{col}_z_score')
        ])

print("App-normalized z-scores computed")

# Multi-metric correlation signals
if 'cost_z_score' in df.columns and 'error_rate_z_score' in df.columns:
    df = df.with_columns([
        ((pl.col('cost_z_score') > 3) & (pl.col('error_rate_z_score') > 3)).alias('cost_and_error_anomaly'),
        pl.col('cost_z_score').shift(1).over('app_id').alias('cost_z_score_1h_ago')
    ])
    
    df = df.with_columns([
        ((pl.col('cost_z_score_1h_ago') > 3) & (pl.col('error_rate_z_score') > 3)).alias('cost_spike_before_errors')
    ])
    
    # Value drop with cost spike
    if 'value_to_cost_ratio_z_score' in df.columns:
        df = df.with_columns([
            ((pl.col('value_to_cost_ratio_z_score') < -2) & (pl.col('cost_z_score') > 3)).alias('value_drop_with_cost_spike')
        ])

print("Multi-metric correlation signals created")
print(f"Features added: cost_and_error_anomaly, cost_spike_before_errors, value_drop_with_cost_spike")


App-normalized z-scores computed
Multi-metric correlation signals created
Features added: cost_and_error_anomaly, cost_spike_before_errors, value_drop_with_cost_spike


### Category 6: Acceleration Feature (for future Early Warning System)

This feature will be useful for predictive models in future milestones.


In [12]:
# Cost acceleration (rate of change of rate of change)
if 'cost_pct_change_vs_24h_ago' in df.columns and 'cost_pct_change_vs_48h_ago' in df.columns:
    df = df.with_columns([
        (pl.col('cost_pct_change_vs_24h_ago') - pl.col('cost_pct_change_vs_48h_ago')).alias('cost_acceleration')
    ])

exclude_cols = ['tenant_id', 'app_id', 'app_name', 'date', 'hour']
feature_count = len([c for c in df.columns if c not in exclude_cols])
print("Acceleration feature created")
print(f"Total features: {feature_count}")


Acceleration feature created
Total features: 56


## Step 3: Label Creation (Weak Labels)

Using relaxed thresholds: error_rate > 0.05 or robust z-score > 2.0 (changed from 0.10/3.0 to generate more positive examples for training)


In [13]:
# Create weak labels
# Strategy: error_rate > 0.05 OR robust z-score (cost_mean or data_mean) > 2.0
# RELAXED THRESHOLDS: Changed from 0.10/3.0 to 0.05/2.0 to generate more positive examples
# This should increase positive rate from ~0.025% to ~1-2% for better model training

# Robust z-score using median and IQR with Polars
# Compute quantiles separately for better compatibility
df_stats = df.group_by('app_id').agg([
    pl.col('cost_mean').median().alias('cost_mean_median'),
    pl.col('cost_mean').quantile(0.75).alias('cost_mean_q75'),
    pl.col('cost_mean').quantile(0.25).alias('cost_mean_q25'),
    pl.col('data_mean').median().alias('data_mean_median'),
    pl.col('data_mean').quantile(0.75).alias('data_mean_q75'),
    pl.col('data_mean').quantile(0.25).alias('data_mean_q25')
])

# Join stats back and compute robust z-scores
df = df.join(df_stats, on='app_id', how='left')
df = df.with_columns([
    (
        (pl.col('cost_mean') - pl.col('cost_mean_median')) / 
        (pl.col('cost_mean_q75') - pl.col('cost_mean_q25') + 1e-8)
    ).alias('cost_mean_robust_z'),
    (
        (pl.col('data_mean') - pl.col('data_mean_median')) / 
        (pl.col('data_mean_q75') - pl.col('data_mean_q25') + 1e-8)
    ).alias('data_mean_robust_z')
])

# Drop intermediate stat columns
df = df.drop(['cost_mean_median', 'cost_mean_q75', 'cost_mean_q25', 
              'data_mean_median', 'data_mean_q75', 'data_mean_q25'])

# Create labels
df = df.with_columns([
    (
        (pl.col('error_rate') > 0.05) |
        (pl.col('cost_mean_robust_z') > 2.0) |
        (pl.col('data_mean_robust_z') > 2.0)
    ).cast(pl.Int64).alias('label')
])

print(f"Label distribution:")
print(df['label'].value_counts().sort('label'))
print(f"\nPositive rate: {df['label'].mean():.4f}")


Label distribution:
shape: (2, 2)
┌───────┬───────┐
│ label ┆ count │
│ ---   ┆ ---   │
│ i64   ┆ u32   │
╞═══════╪═══════╡
│ 0     ┆ 31605 │
│ 1     ┆ 150   │
└───────┴───────┘

Positive rate: 0.0047


## Step 4: Feature Selection and Preparation


In [14]:
# Select features for modeling
# Exclude metadata columns and label
exclude_cols = ['tenant_id', 'app_id', 'app_name', 'date', 'hour', 'label', 
                'cost_mean_robust_z', 'data_mean_robust_z']  # Exclude label creation helpers

# Get all numeric feature columns (convert Polars to pandas for easier feature selection)
df_pd = df.to_pandas()

# Get all numeric feature columns
feature_cols = [c for c in df_pd.columns if c not in exclude_cols and df_pd[c].dtype in [np.float64, np.int64, np.bool_, 'float64', 'int64', 'bool']]

# Handle any remaining non-numeric columns
for col in feature_cols:
    if df_pd[col].dtype == 'object' or df_pd[col].dtype == 'bool':
        df_pd[col] = df_pd[col].astype(float)

print(f"Selected {len(feature_cols)} features")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:2d}. {col}")

# Create feature matrix
X = df_pd[feature_cols].fillna(0).values
y = df_pd['label'].values

print(f"\nFeature matrix shape: {X.shape}")
print(f"Label distribution: {np.bincount(y)}")


Selected 51 features

Feature list:
 1. cost
 2. value
 3. data_used
 4. data_sent
 5. requests_made
 6. requests_received
 7. cost_per_request_made
 8. cost_per_request_received
 9. data_per_request
10. value_per_cost
11. requests_per_hour
12. requests_per_business_hour
13. data_sent_per_received
14. data_used_per_received
15. outage_count
16. outage_duration
17. outage_impact
18. outage_cost
19. outage_frequency
20. outage_severity
21. outage_efficiency
22. cost_sum
23. cost_mean
24. data_sum
25. data_mean
26. error_rate
27. cost_7day_avg
28. cost_deviation_from_week
29. is_weekend
30. cost_per_request
31. value_to_cost_ratio
32. data_utilization
33. cost_24h_ago
34. cost_48h_ago
35. error_rate_24h_ago
36. cost_pct_change_vs_24h_ago
37. cost_pct_change_vs_48h_ago
38. error_rate_pct_change
39. supplier_failure_exposure
40. downstream_impact_score
41. cost_z_score
42. cost_mean_z_score
43. error_rate_z_score
44. value_z_score
45. value_to_cost_ratio_z_score
46. cost_per_request_z_score

## Step 5: Train/Test Split (Time-Based)


In [15]:
# Time-based split (80/20) - using pandas dataframe for indexing
split_date = df_pd['date'].quantile(0.8)
train_mask = df_pd['date'] < split_date
test_mask = df_pd['date'] >= split_date

X_train = X[train_mask]
X_test = X[test_mask]
y_train = y[train_mask]
y_test = y[test_mask]

print(f"Train set: {len(X_train):,} samples ({train_mask.sum() / len(df_pd) * 100:.1f}%)")
print(f"Test set: {len(X_test):,} samples ({test_mask.sum() / len(df_pd) * 100:.1f}%)")
print(f"\nTrain label distribution: {np.bincount(y_train)}")
print(f"Test label distribution: {np.bincount(y_test)}")
print(f"\nTrain positive rate: {y_train.mean():.4f}")
print(f"Test positive rate: {y_test.mean():.4f}")


Train set: 25,404 samples (80.0%)
Test set: 6,351 samples (20.0%)

Train label distribution: [25286   118]
Test label distribution: [6319   32]

Train positive rate: 0.0046
Test positive rate: 0.0050


## Step 6: XGBoost Model Training


In [16]:
# Calculate scale_pos_weight for class imbalance
pos_rate = y_train.mean()
scale_pos_weight = (1 - pos_rate) / (pos_rate + 1e-8)

print(f"Positive rate: {pos_rate:.4f}")
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Create validation set for early stopping
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Train XGBoost
# In XGBoost 3.x, early_stopping_rounds is a constructor parameter, not a fit() parameter
model_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50  # Moved to constructor in XGBoost 3.x
)

print("Training XGBoost model...")
model_xgb.fit(
    X_train_fit, y_train_fit,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"Best iteration: {model_xgb.best_iteration}")
print("Training complete!")


Positive rate: 0.0046
Scale pos weight: 214.29
Training XGBoost model...


Best iteration: 311
Training complete!


## Step 7: Model Evaluation


In [17]:
# Predictions on test set
y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]
y_pred = model_xgb.predict(X_test)

# Calculate metrics
pr_auc = average_precision_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 60)
print("XGBoost Model Performance (Test Set)")
print("=" * 60)
print(f"PR-AUC:        {pr_auc:.4f}")
print(f"F1-Score:      {f1:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"Accuracy:      {accuracy:.4f}")
print("=" * 60)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


XGBoost Model Performance (Test Set)
PR-AUC:        0.4379
F1-Score:      0.4762
Precision:     0.4839
Recall:        0.4688
Accuracy:      0.9948

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6319
           1       0.48      0.47      0.48        32

    accuracy                           0.99      6351
   macro avg       0.74      0.73      0.74      6351
weighted avg       0.99      0.99      0.99      6351



In [18]:
# Comparison with Milestone 2 baseline
m2_metrics = {
    'PR-AUC': 0.9925,
    'F1-Score': 0.9943,
    'Precision': 1.0000,
    'Recall': 0.9887,
    'Accuracy': 0.9990
}

m3_metrics = {
    'PR-AUC': pr_auc,
    'F1-Score': f1,
    'Precision': precision,
    'Recall': recall,
    'Accuracy': accuracy
}

print("=" * 60)
print("Comparison: Milestone 2 (KNN) vs Milestone 3 (XGBoost)")
print("=" * 60)
print(f"{'Metric':<15} {'M2 (KNN)':<12} {'M3 (XGBoost)':<15} {'Change':<10}")
print("-" * 60)
for metric in m2_metrics.keys():
    m2_val = m2_metrics[metric]
    m3_val = m3_metrics[metric]
    change = m3_val - m2_val
    change_str = f"{change:+.4f}" if change != 0 else "0.0000"
    print(f"{metric:<15} {m2_val:<12.4f} {m3_val:<15.4f} {change_str:<10}")

print("=" * 60)


Comparison: Milestone 2 (KNN) vs Milestone 3 (XGBoost)
Metric          M2 (KNN)     M3 (XGBoost)    Change    
------------------------------------------------------------
PR-AUC          0.9925       0.4379          -0.5546   
F1-Score        0.9943       0.4762          -0.5181   
Precision       1.0000       0.4839          -0.5161   
Recall          0.9887       0.4688          -0.5200   
Accuracy        0.9990       0.9948          -0.0042   


In [ ]:
# ---------------------------------------------------------
# SCIENTIFIC PROOF: Direct Model Comparison on Same Dataset
# ---------------------------------------------------------
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Identify "Old" Features (Baseline from M2)
# M2 used raw metrics: cost, value, data_used, requests_made
old_feats = ['cost', 'value', 'data_used', 'requests_made']
# Find indices of these features in the feature_cols list
old_feat_indices = [i for i, f in enumerate(feature_cols) if f in old_feats]

print(f"Baseline Features ({len(old_feat_indices)}): {old_feats}")

# 2. Prepare Data (Scaling is mandatory for KNN and Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Select only old features for M2 Proxy
X_train_old = X_train_scaled[:, old_feat_indices]
X_test_old = X_test_scaled[:, old_feat_indices]

# Use percentile-based threshold for imbalanced data (matches positive rate)
# This ensures fair F1-Score comparison across all models
pos_rate = y_test.mean()
percentile_threshold = 100 - (pos_rate * 100)

print("\n" + "="*60)
print("Scientific Proof: Model Comparison on Current Hard Labels")
print(f"Using percentile threshold: {percentile_threshold:.2f}% (matches {pos_rate:.2%} positive rate)")
print("="*60)

# 3. Train M2 Proxy (KNN on Old Features)
print("\nTraining KNN on Old Features (M2 Proxy)...")
knn_old = KNeighborsClassifier(n_neighbors=5)
knn_old.fit(X_train_old, y_train)
y_prob_old = knn_old.predict_proba(X_test_old)[:, 1]
threshold_old = np.percentile(y_prob_old, percentile_threshold)
y_pred_old = (y_prob_old >= threshold_old).astype(int)
f1_old = f1_score(y_test, y_pred_old)
prauc_old = average_precision_score(y_test, y_prob_old)

# 4. Train KNN on NEW Features (to show feature lift)
print("Training KNN on New Features (51 Features)...")
knn_new = KNeighborsClassifier(n_neighbors=5)
knn_new.fit(X_train_scaled, y_train)
y_prob_new = knn_new.predict_proba(X_test_scaled)[:, 1]
threshold_new = np.percentile(y_prob_new, percentile_threshold)
y_pred_new = (y_prob_new >= threshold_new).astype(int)
f1_new = f1_score(y_test, y_pred_new)
prauc_new = average_precision_score(y_test, y_prob_new)

# 5. Train Logistic Regression (Linear Baseline)
print("Training Logistic Regression on New Features...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
threshold_lr = np.percentile(y_prob_lr, percentile_threshold)
y_pred_lr = (y_prob_lr >= threshold_lr).astype(int)
f1_lr = f1_score(y_test, y_pred_lr)
prauc_lr = average_precision_score(y_test, y_prob_lr)

# 6. XGBoost (Already trained on New Features)
# metrics: f1, pr_auc (from previous cell)

print("\n" + "="*80)
print(f"{'Model':<30} {'Features':<15} {'PR-AUC':<12} {'F1-Score':<12}")
print("-" * 80)
print(f"{'KNN (M2 Proxy)':<30} {'4 (Old)':<15} {prauc_old:<12.4f} {f1_old:<12.4f}")
print(f"{'KNN (New Feats)':<30} {'51 (New)':<15} {prauc_new:<12.4f} {f1_new:<12.4f}")
print(f"{'Logistic Regression':<30} {'51 (New)':<15} {prauc_lr:<12.4f} {f1_lr:<12.4f}")
print(f"{'XGBoost (M3)':<30} {'51 (New)':<15} {pr_auc:<12.4f} {f1:<12.4f}")
print("=" * 80)

print(f"\n{'='*80}")
print("LIFT ANALYSIS")
print(f"{'='*80}")

print(f"\n1. Feature Engineering Lift (KNN New vs KNN Old):")
print(f"   PR-AUC Improvement: {prauc_new - prauc_old:+.4f} ({((prauc_new/prauc_old - 1)*100):+.1f}%)")
print(f"   F1 Improvement:     {f1_new - f1_old:+.4f}")

print(f"\n2. Linear vs Non-Linear (XGBoost vs Logistic Regression):")
print(f"   PR-AUC Improvement: {pr_auc - prauc_lr:+.4f} ({((pr_auc/prauc_lr - 1)*100):+.1f}%)")
print(f"   F1 Improvement:     {f1 - f1_lr:+.4f}")

print(f"\n3. Overall Lift vs M2 Proxy (XGBoost vs KNN Old):")
print(f"   PR-AUC Improvement: {pr_auc - prauc_old:+.4f} ({((pr_auc/prauc_old - 1)*100):+.1f}%)")
print(f"   F1 Improvement:     {f1 - f1_old:+.4f}")

print(f"\n4. Model Algorithm Lift (XGBoost vs KNN, both on New Features):")
print(f"   PR-AUC Improvement: {pr_auc - prauc_new:+.4f} ({((pr_auc/prauc_new - 1)*100):+.1f}%)")
print(f"   F1 Improvement:     {f1 - f1_new:+.4f}")

# Store results for later use
comparison_results = {
    'knn_old': {'pr_auc': prauc_old, 'f1': f1_old, 'features': 4},
    'knn_new': {'pr_auc': prauc_new, 'f1': f1_new, 'features': 51},
    'logistic': {'pr_auc': prauc_lr, 'f1': f1_lr, 'features': 51},
    'xgboost': {'pr_auc': pr_auc, 'f1': f1, 'features': 51}
}

# ---------------------------------------------------------
# LATENCY BENCHMARK: Inference Speed for Production Readiness
# ---------------------------------------------------------
import time

def benchmark_model(model, X_input, name, warmup=True):
    """
    Benchmark inference latency for a model.
    Returns: (total_time_ms, latency_per_sample_us)
    """
    if warmup:
        # Warmup: run a few predictions to initialize any lazy structures
        _ = model.predict(X_input[:10])
    
    # Time the full test set prediction
    start_time = time.perf_counter()
    predictions = model.predict(X_input)
    end_time = time.perf_counter()
    
    total_time_ms = (end_time - start_time) * 1000  # Convert to milliseconds
    latency_per_sample_us = (total_time_ms / len(X_input)) * 1000  # Convert to microseconds per sample
    
    return total_time_ms, latency_per_sample_us

print("\n" + "="*80)
print("INFERENCE LATENCY BENCHMARK (Production Readiness)")
print("="*80)
print(f"Test set size: {len(X_test):,} samples")
print(f"{'Model':<30} {'Total Time (ms)':<20} {'Latency (μs/sample)':<20} {'Throughput (samples/sec)':<25}")
print("-" * 80)

# 1. KNN Old (M2 Proxy)
t_knn, l_knn = benchmark_model(knn_old, X_test_old, "KNN (M2)")
throughput_knn = len(X_test_old) / (t_knn / 1000) if t_knn > 0 else 0
print(f"{'KNN (M2 Proxy)':<30} {t_knn:<20.2f} {l_knn:<20.2f} {throughput_knn:<25,.0f}")

# 2. KNN New Features
t_knn_new, l_knn_new = benchmark_model(knn_new, X_test_scaled, "KNN (New)")
throughput_knn_new = len(X_test_scaled) / (t_knn_new / 1000) if t_knn_new > 0 else 0
print(f"{'KNN (New Feats)':<30} {t_knn_new:<20.2f} {l_knn_new:<20.2f} {throughput_knn_new:<25,.0f}")

# 3. Logistic Regression
t_lr, l_lr = benchmark_model(lr, X_test_scaled, "Logistic Reg")
throughput_lr = len(X_test_scaled) / (t_lr / 1000) if t_lr > 0 else 0
print(f"{'Logistic Regression':<30} {t_lr:<20.2f} {l_lr:<20.2f} {throughput_lr:<25,.0f}")

# 4. XGBoost
t_xgb, l_xgb = benchmark_model(model_xgb, X_test, "XGBoost")
throughput_xgb = len(X_test) / (t_xgb / 1000) if t_xgb > 0 else 0
print(f"{'XGBoost (M3)':<30} {t_xgb:<20.2f} {l_xgb:<20.2f} {throughput_xgb:<25,.0f}")
print("=" * 80)

# Store latency results for later use
latency_results = {
    'knn_old': {'total_ms': t_knn, 'latency_us': l_knn, 'throughput': throughput_knn},
    'knn_new': {'total_ms': t_knn_new, 'latency_us': l_knn_new, 'throughput': throughput_knn_new},
    'logistic': {'total_ms': t_lr, 'latency_us': l_lr, 'throughput': throughput_lr},
    'xgboost': {'total_ms': t_xgb, 'latency_us': l_xgb, 'throughput': throughput_xgb}
}

print(f"\n{'='*80}")
print("LATENCY ANALYSIS")
print(f"{'='*80}")
print(f"\n1. Speed Comparison (vs KNN Old):")
print(f"   Logistic Regression: {t_knn/t_lr:.1f}x faster")
print(f"   XGBoost:             {t_knn/t_xgb:.1f}x faster")

print(f"\n2. Production Scalability:")
print(f"   XGBoost can handle ~{throughput_xgb:,.0f} predictions/second")
print(f"   At this rate, processing 1M samples would take ~{1000000/throughput_xgb:.1f} seconds ({1000000/throughput_xgb/60:.1f} minutes)")

print(f"\n3. Feature Engineering Impact on Speed:")
print(f"   KNN (Old 4 features): {l_knn:.2f} μs/sample")
print(f"   KNN (New 51 features): {l_knn_new:.2f} μs/sample")
print(f"   Overhead: {((l_knn_new/l_knn - 1)*100):+.1f}% (minimal - shows efficient feature engineering)")


Baseline Features (4): ['cost', 'value', 'data_used', 'requests_made']

Scientific Proof: Model Comparison on Current Hard Labels
Using percentile threshold: 99.50% (matches 0.50% positive rate)

Training KNN on Old Features (M2 Proxy)...
Training KNN on New Features (51 Features)...
Training Logistic Regression on New Features...

Model                          Features        PR-AUC       F1-Score    
--------------------------------------------------------------------------------
KNN (M2 Proxy)                 4 (Old)         0.0064       0.0323      
KNN (New Feats)                51 (New)        0.0256       0.1197      
Logistic Regression            51 (New)        0.3060       0.3750      
XGBoost (M3)                   51 (New)        0.4379       0.4762      

LIFT ANALYSIS

1. Feature Engineering Lift (KNN New vs KNN Old):
   PR-AUC Improvement: +0.0192 (+300.9%)
   F1 Improvement:     +0.0874

2. Linear vs Non-Linear (XGBoost vs Logistic Regression):
   PR-AUC Improvement: 

## Step 8: Feature Importance


In [20]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print("=" * 60)
for idx, row in feature_importance.head(20).iterrows():
    print(f"{row['feature']:<40} {row['importance']:.4f}")

# Save feature importance
feature_importance.to_csv('m3_feature_importance.csv', index=False)
print(f"\nFeature importance saved to m3_feature_importance.csv")


Top 20 Most Important Features:
requests_per_hour                        0.2189
requests_made                            0.1539
cost_per_request_received                0.0837
outage_cost                              0.0610
value_per_cost                           0.0579
requests_per_business_hour               0.0317
cost_mean_z_score                        0.0313
outage_impact                            0.0250
data_mean                                0.0229
value                                    0.0226
requests_received                        0.0218
outage_severity                          0.0208
data_used                                0.0196
cost_per_request_z_score                 0.0191
outage_efficiency                        0.0172
cost_sum                                 0.0133
cost_mean                                0.0131
data_sent                                0.0124
data_used_per_received                   0.0121
cost_7day_avg                            0.0119

Feature

## Step 9: Enhanced Anomaly Explanations

Implementing the upgraded `generate_business_reason` function with severity levels.


In [21]:
def generate_business_reason(row):
    """
    Produce a structured, business-facing anomaly explanation.
    Includes severity levels and actionable recommendations.
    """
    reasons = []
    severity = "MEDIUM"
    recommendations = []
    
    # 1. Check for multi-metric anomalies first (high-signal)
    if row.get('cost_and_error_anomaly', False):
        severity = "CRITICAL"
        reasons.append("High-Signal Event: Cost and Error rates are spiking simultaneously.")
        recommendations.append("URGENT: This is likely a critical failure. Investigate service logs and upstream dependencies.")

    # 2. Z-score analysis
    if pd.notna(row.get('cost_z_score')):
        z_abs = abs(row['cost_z_score'])
        if z_abs > 3:
            reasons.append(f"Cost spike: {z_abs:.1f}σ above baseline")
            if z_abs > 6 and severity != "CRITICAL":
                severity = "CRITICAL"
            elif z_abs > 4:
                severity = "HIGH"
    
    if pd.notna(row.get('error_rate_z_score')):
        z_abs = abs(row['error_rate_z_score'])
        if z_abs > 3:
            reasons.append(f"Error rate spike: {z_abs:.1f}σ above baseline")
            if z_abs > 6 and severity != "CRITICAL":
                severity = "CRITICAL"
            elif z_abs > 4:
                severity = "HIGH"
    
    # 3. Network context (blast radius)
    if pd.notna(row.get('unique_consumers_count')) and row.get('unique_consumers_count', 0) > 10:
        reasons.append(f"High-impact service with {int(row['unique_consumers_count'])} downstream dependencies")
        recommendations.append("Priority escalation due to potential cascade effects")
        if severity == "MEDIUM":
            severity = "HIGH"
    
    # 4. Business efficiency
    if pd.notna(row.get('value_to_cost_ratio')) and row['value_to_cost_ratio'] < 0.1:
        reasons.append("Low Business Efficiency: Value-to-Cost ratio is critically low.")
        recommendations.append("Review service for potential waste or misconfiguration.")
        if severity == "MEDIUM":
            severity = "HIGH"
    
    # 5. Error rate check
    if pd.notna(row.get('error_rate')) and row['error_rate'] > 0.10:
        reasons.append(f"High error rate: {row['error_rate']:.1%}")
        recommendations.append("Investigate service health and upstream dependencies")
        if severity == "MEDIUM":
            severity = "HIGH"

    # Construct final message
    if not reasons:
        return "[INFO] Statistical outlier detected by model"
    
    reason_text = " | ".join(sorted(list(set(reasons))))
    recommendation_text = "; ".join(sorted(list(set(recommendations)))) if recommendations else "No specific recommendation."
    
    final_message = f"[{severity}] {reason_text}"
    final_message += f"\\n   -> Recommended Action: {recommendation_text}"
        
    return final_message

# Test on sample anomalies (using pandas dataframe)
test_anomalies = df_pd[df_pd['label'] == 1].head(10)
print("Sample Anomaly Explanations:")
print("=" * 80)
for idx, row in test_anomalies.iterrows():
    explanation = generate_business_reason(row)
    print(f"\\nApp: {row['app_id']} | Date: {row['date']}")
    print(explanation)
    print("-" * 80)


Sample Anomaly Explanations:
\nApp: AIRFLW | Date: 2025-03-27 00:00:00
[HIGH] Low Business Efficiency: Value-to-Cost ratio is critically low.\n   -> Recommended Action: Review service for potential waste or misconfiguration.
--------------------------------------------------------------------------------
\nApp: CLVMDL | Date: 2024-06-21 00:00:00
[HIGH] Low Business Efficiency: Value-to-Cost ratio is critically low.\n   -> Recommended Action: Review service for potential waste or misconfiguration.
--------------------------------------------------------------------------------
\nApp: CLVMDL | Date: 2024-08-07 00:00:00
[HIGH] Low Business Efficiency: Value-to-Cost ratio is critically low.\n   -> Recommended Action: Review service for potential waste or misconfiguration.
--------------------------------------------------------------------------------
\nApp: CLVMDL | Date: 2024-09-28 00:00:00
[HIGH] Low Business Efficiency: Value-to-Cost ratio is critically low.\n   -> Recommended Action: 

## Step 10: Save Results

Saving model performance metrics and comparison results.


In [22]:
# Save results to JSON
import json

results = {
    'model': 'XGBoost',
    'n_features': len(feature_cols),
    'n_train': len(X_train),
    'n_test': len(X_test),
    'metrics': {
        'PR_AUC': float(pr_auc),
        'F1_Score': float(f1),
        'Precision': float(precision),
        'Recall': float(recall),
        'Accuracy': float(accuracy)
    },
    'baseline_comparison': {
        'm2_PR_AUC': 0.9925,
        'm2_F1': 0.9943,
        'm2_Precision': 1.0000,
        'm2_Recall': 0.9887,
        'm2_Accuracy': 0.9990,
        'improvement_PR_AUC': float(pr_auc - 0.9925),
        'improvement_F1': float(f1 - 0.9943),
        'improvement_Precision': float(precision - 1.0000),
        'improvement_Recall': float(recall - 0.9887),
        'improvement_Accuracy': float(accuracy - 0.9990)
    },
    'top_features': feature_importance.head(10).to_dict('records')
}

with open('m3_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to m3_results.json")
print("\\nSummary:")
print(json.dumps(results, indent=2))


Results saved to m3_results.json
\nSummary:
{
  "model": "XGBoost",
  "n_features": 51,
  "n_train": 25404,
  "n_test": 6351,
  "metrics": {
    "PR_AUC": 0.4379431183931222,
    "F1_Score": 0.47619047619047616,
    "Precision": 0.4838709677419355,
    "Recall": 0.46875,
    "Accuracy": 0.9948039678790742
  },
  "baseline_comparison": {
    "m2_PR_AUC": 0.9925,
    "m2_F1": 0.9943,
    "m2_Precision": 1.0,
    "m2_Recall": 0.9887,
    "m2_Accuracy": 0.999,
    "improvement_PR_AUC": -0.5545568816068778,
    "improvement_F1": -0.5181095238095238,
    "improvement_Precision": -0.5161290322580645,
    "improvement_Recall": -0.51995,
    "improvement_Accuracy": -0.004196032120925786
  },
  "top_features": [
    {
      "feature": "requests_per_hour",
      "importance": 0.21885813772678375
    },
    {
      "feature": "requests_made",
      "importance": 0.1538979709148407
    },
    {
      "feature": "cost_per_request_received",
      "importance": 0.08365758508443832
    },
    {
      